In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost
from sklearn.metrics import classification_report, confusion_matrix, recall_score, make_scorer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
#data
df = pd.read_parquet('...')  #pfad anpassen

In [ ]:
#train-test-split
label_col = 'target'
X = df.drop(columns=label_col)
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
#compute class weights
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

#further boost for class 2
weight_boost = {0: 1.0, 1: 1.5, 2: 3.0}
class_weights = {cls: w * weight_boost[cls] for cls, w in zip(classes, weights)}
print("Class Weights:", class_weights)

#Sample Weights
sample_weights = y_train.map(class_weights)

In [ ]:
#GridSearch Parameter
param_grid = {
    'learning_rate': [0.03, 0.05],
    'max_depth': [5, 6],
    'n_estimators': [100, 150],
    'colsample_bytree': [0.7, 0.9],
    'gamma': [0.2, 0.4]
}

#base model
base_model = xgboost.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_jobs=6,
    random_state=0,
    verbosity=0
)

#GridSearchCV with Recall-optimization
recall_scorer = make_scorer(recall_score, average='macro')

grid_model = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring=recall_scorer,
    cv=3,
    verbose=2,
    n_jobs=-1
)


In [ ]:
#train model
grid_model.fit(X_train, y_train, sample_weight=sample_weights)
best_model = grid_model.best_estimator_
print("Best Params:", grid_model.best_params_)

In [ ]:
#prediction with tuned thresholds
y_proba = best_model.predict_proba(X_test)

thresholds = {
    0: 0.6,  # no error
    1: 0.25,  # suspect
    2: 0.15  # error
}

y_pred_thresh = []
for row in y_proba:
    if row[2] > thresholds[2]:
        y_pred_thresh.append(2)
    elif row[1] > thresholds[1]:
        y_pred_thresh.append(1)
    else:
        y_pred_thresh.append(0)

In [ ]:
#evaluation
print("\nClassification Report:\n", classification_report(y_test, y_pred_thresh))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_thresh))

cm = confusion_matrix(y_test, y_pred_thresh)
labels = ['no error (0)', 'suspect (1)', 'error (2)']

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix – Recall Optimized XGBoost')
plt.tight_layout()
plt.show()